In [0]:
# Define your project variables globally
catalog = "capstone_101"
 
# Verify the catalog exists in your session
print(f"Active Catalog: {catalog}")

In [0]:
# Create a text widget
dbutils.widgets.text("catalog_name", "capstone_101", "Catalog Name")
 
# Get the value from the widget
catalog = dbutils.widgets.get("catalog_name")

In [0]:
from pyspark.sql.functions import col, sum, round
 
silver_df = spark.read.table(f"{catalog}.silver.cleaned_sales")

# Table: total_revenue_by_country
(silver_df
    .groupBy("Country")
    .agg(round(sum(col("UnitPrice").cast("double") * col("Quantity").cast("int")), 2).alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
    .write.mode("overwrite")
    .saveAsTable(f"{catalog}.gold.total_revenue_by_country"))

# Table: high_value_customers
(silver_df
    .groupBy("CustomerID")
    .agg(round(sum(col("UnitPrice").cast("double") * col("Quantity").cast("int")), 2).alias("total_spend"))
    .filter(col("total_spend") > 5000)
    .orderBy(col("total_spend").desc())
    .write.mode("overwrite")
    .saveAsTable(f"{catalog}.gold.high_value_customers"))